In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install datasets transformers tokenizers sentencepiece tqdm accelerate

Looking in indexes: https://download.pytorch.org/whl/cu121


In [2]:
import torch
import re
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm
from datasets import load_dataset, interleave_datasets
from transformers import LlamaTokenizer
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} (40GB VRAM Detected)")

# Scale up to 1 million samples per language
num_samples = 1000000
print("Streaming large-scale datasets...")

ds_hi = load_dataset("ai4bharat/samanantar", "hi", split="train", streaming=True).take(num_samples)
ds_bn = load_dataset("ai4bharat/samanantar", "bn", split="train", streaming=True).take(num_samples)

def clean_text(example):
    # Ensure there's a clear space between languages
    src = example['src'].strip()
    tgt = example['tgt'].strip()
    text = f"{src} {tgt}"
    # Replace multiple spaces with one, but keep the space!
    text = re.sub(r"\s+", " ", text)
    return {"text": text}

# Interleave and Shuffle with a 10k buffer for the A100
dataset = interleave_datasets(
    [ds_hi, ds_bn],
    stopping_strategy="all_exhausted"
).shuffle(seed=42, buffer_size=10000).map(clean_text)

print("Dataset stream initialized and shuffled.")

Using device: cuda (40GB VRAM Detected)
Streaming large-scale datasets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Dataset stream initialized and shuffled.


In [5]:
import os
import sentencepiece as spm
from transformers import LlamaTokenizer

# 1. CREATE THE DATA FILE (The missing piece)
print("Creating vocab_train.txt from dataset...")
with open('vocab_train.txt', 'w', encoding='utf-8') as f:
    # Take a representative sample for the vocabulary
    for i, example in enumerate(dataset.take(50000)):
        f.write(example['text'] + "\n")
        if i % 10000 == 0:
            print(f"Written {i} samples...")

# 2. Train SentencePiece
print("Training SentencePiece model (32k vocab)...")
spm.SentencePieceTrainer.train(
    input='vocab_train.txt',
    model_prefix='indic_gpt_pro',
    vocab_size=32000,
    model_type='unigram',
    character_coverage=1.0,
    byte_fallback=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)

# 3. VERIFICATION
sp = spm.SentencePieceProcessor(model_file='indic_gpt_pro.model')
actual_spm_size = sp.get_piece_size()
print(f"Internal SentencePiece size: {actual_spm_size}")

# 4. LOAD TOKENIZER
tokenizer = LlamaTokenizer(
    vocab_file="indic_gpt_pro.model",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    legacy=True,
    use_fast=False
)

print(f"Verified Vocab size: {len(tokenizer)}")
if len(tokenizer) > 30000:
    print("Success! Your model is now safe to build.")

Creating vocab_train.txt from dataset...
Written 0 samples...
Written 10000 samples...
Written 20000 samples...
Written 30000 samples...
Written 40000 samples...
Training SentencePiece model (32k vocab)...
Internal SentencePiece size: 32000
Verified Vocab size: 4


In [6]:
class GPTModel(nn.Module):
    def __init__(self, vocab_size=32000, n_layers=12, hidden_size=768, n_heads=12, context_length=512):
        super().__init__()
        self.context_length = context_length
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.pos_encoder = nn.Embedding(context_length, hidden_size)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_size,
            nhead=n_heads,
            dim_feedforward=hidden_size * 4,
            batch_first=True,
            norm_first=True,
            activation="gelu",
            dropout=0.1
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device)
        h = self.embed(x) + self.pos_encoder(torch.arange(seq_len, device=x.device).unsqueeze(0))
        h = self.transformer_decoder(tgt=h, tgt_mask=tgt_mask, memory=h)
        return self.lm_head(self.norm(h))

model = GPTModel(vocab_size=len(tokenizer)).to(device)


In [7]:
from torch.cuda.amp import GradScaler, autocast

# 1. Hardware Optimization
# Emptying cache to start fresh for the larger batch

torch.cuda.empty_cache()
MAX_LEN = 512

# Increased batch_size to 128 to utilize the 40GB A100 VRAM
batch_size = 128

# 2. Refined Optimization Setup
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.1)
scaler = GradScaler()
criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)

def get_streaming_batches(stream_ds, batch_size, tokenizer, max_len):
    batch = []
    # Larger local buffer to keep the GPU fed
    for example in stream_ds:
        tokens = tokenizer(example["text"], truncation=True, padding="max_length",
                          max_length=max_len, return_tensors="pt")
        batch.append(tokens["input_ids"].squeeze(0))
        if len(batch) == batch_size:
            yield torch.stack(batch)
            batch = []

print(f"Launching Max-Utilization Training on A100...")
print(f"Targeting ~35GB/40GB VRAM usage with batch_size={batch_size}")

model.train()
max_steps = 5000
pbar = tqdm(total=max_steps)

for step, inputs in enumerate(get_streaming_batches(dataset, batch_size, tokenizer, MAX_LEN)):
    inputs = inputs.to(device, non_blocking=True) # non_blocking for speed
    optimizer.zero_grad(set_to_none=True) # More efficient than zero_grad()

    with autocast():
        logits = model(inputs[:, :-1])
        targets = inputs[:, 1:]
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

    scaler.scale(loss).backward()

    # Gradient clipping to prevent 'nan'
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    scaler.step(optimizer)
    scaler.update()

    pbar.update(1)
    if step % 50 == 0:
        pbar.set_description(f"Loss: {loss.item():.4f}")

    if step >= max_steps:
        break

pbar.close()
torch.save(model.state_dict(), "indic_gpt_pro_ultra.pt")
print("High-performance training complete!")

/tmp/ipykernel_4796/4000121050.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Launching Max-Utilization Training on A100...
Targeting ~35GB/40GB VRAM usage with batch_size=128


  0%|          | 0/5000 [00:00<?, ?it/s]/tmp/ipykernel_4796/4000121050.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Loss: 0.3488: : 5001it [51:10,  1.63it/s]


High-performance training complete!


In [8]:
torch.save(model.state_dict(), "indic_gpt_pro.pt")

In [13]:
def generate_pro(prompt, max_new_tokens=100, temperature=0.8, top_p=0.9):
    model.eval()
    # Tokenize and move to device
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # Safety Check: Prevent the 'IndexError: index -1' if prompt is empty
    if input_ids.shape[1] == 0:
        return "Error: Tokenizer produced empty input. Check your prompt."

    for _ in range(max_new_tokens):
        with torch.no_grad():
            # Generate logits for the last token
            logits = model(input_ids)[:, -1, :] / temperature

            # Top-P (Nucleus) Sampling
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[0, indices_to_remove] = -float('Inf')

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

print("generate_pro function defined and ready!")

generate_pro function defined and ready!


In [14]:
# Final Performance Check
print("--- 🇮🇳 Refined Hindi Test ---")
print(generate_pro("भारत एक", temperature=0.7, top_p=0.9))

print("\n--- 🇧🇩 Refined Bengali Test ---")
print(generate_pro("ভারত একটি", temperature=0.7, top_p=0.9))

print("\n--- Economy Test ---")
print(generate_pro("The Indian economy is", temperature=0.7, top_p=0.9))

--- 🇮🇳 Refined Hindi Test ---
Error: Tokenizer produced empty input. Check your prompt.

--- 🇧🇩 Refined Bengali Test ---
Error: Tokenizer produced empty input. Check your prompt.

--- 📈 Economy Test ---
Error: Tokenizer produced empty input. Check your prompt.
